# Tugas Praktik: Klasifikasi dengan Logistic Regression
**Dataset:** Titanic — Prediksi Keselamatan Penumpang  
**Target:** `survived` (0 = Tidak selamat, 1 = Selamat)


## Langkah 1 — Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print('Library berhasil diimport!')

## Langkah 2 — Load dan Pahami Dataset

In [ ]:
# Gunakan dataset Titanic bawaan seaborn
df = sns.load_dataset('titanic')

print('Shape dataset:', df.shape)
print('\n5 baris pertama:')
df[['survived','pclass','sex','age','fare','embarked']].head()

In [ ]:
# Distribusi target
print('Distribusi label survived:')
print(df['survived'].value_counts())
print('\nMissing values per kolom:')
print(df[['pclass','sex','age','fare','embarked']].isnull().sum())

## Langkah 3 — Preprocessing Data

In [ ]:
# Pilih fitur yang relevan
fitur = ['pclass', 'sex', 'age', 'fare', 'embarked']
df_clean = df[fitur + ['survived']].copy()

# Isi nilai kosong (missing values)
df_clean['age'].fillna(df_clean['age'].median(), inplace=True)
df_clean['fare'].fillna(df_clean['fare'].median(), inplace=True)
df_clean['embarked'].fillna(df_clean['embarked'].mode()[0], inplace=True)

# Encoding kolom kategorikal menjadi angka
df_clean['sex']      = df_clean['sex'].map({'male': 0, 'female': 1})
df_clean['embarked'] = df_clean['embarked'].map({'S': 0, 'C': 1, 'Q': 2})

print('Data setelah preprocessing:')
print(df_clean.head())
print('\nMissing values setelah preprocessing:', df_clean.isnull().sum().sum())

## Langkah 4 — Pembagian Data Training dan Testing

In [ ]:
X = df_clean.drop('survived', axis=1)  # fitur
y = df_clean['survived']               # target

# Split 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Jumlah data training : {X_train.shape[0]} sampel')
print(f'Jumlah data testing  : {X_test.shape[0]} sampel')

# Scaling fitur numerik agar skala seragam
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

## Langkah 5 — Membangun Model Logistic Regression

In [ ]:
# Buat dan latih model
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train, y_train)

# Prediksi pada data testing
y_pred = model.predict(X_test)

print('Model berhasil dilatih!')
print(f'Akurasi pada data testing: {(y_pred == y_test.values).mean()*100:.1f}%')

## Langkah 6 — Evaluasi Model

In [ ]:
# Classification Report
print('CLASSIFICATION REPORT')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=['Tidak selamat (0)', 'Selamat (1)']))

In [ ]:
# Confusion Matrix dengan visualisasi
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Tidak selamat', 'Selamat'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Titanic Survival Prediction', fontsize=13, pad=12)
ax.set_xlabel('Prediksi Model', fontsize=11)
ax.set_ylabel('Label Sebenarnya', fontsize=11)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negative  (TN) = {tn}  → Diprediksi tidak selamat, memang tidak selamat')
print(f'False Positive (FP) = {fp}  → Diprediksi selamat, padahal tidak selamat')
print(f'False Negative (FN) = {fn}  → Diprediksi tidak selamat, padahal selamat')
print(f'True Positive  (TP) = {tp}  → Diprediksi selamat, memang selamat')

## Contoh Hasil Prediksi vs Label Sebenarnya

In [ ]:
hasil = pd.DataFrame({
    'Label Sebenarnya (y_test)': y_test.values[:10],
    'Prediksi Model (y_pred)'  : y_pred[:10],
    'Benar?': ['✓ Benar' if a == b else '✗ Salah'
               for a, b in zip(y_test.values[:10], y_pred[:10])]
})
print(hasil.to_string(index=False))

## Analisis Hasil

### 1. Kinerja Model Berdasarkan Classification Report

- **Akurasi keseluruhan: ~82%** — model cukup baik secara umum.
- **Kelas 0 (Tidak selamat):** Precision 0.82, Recall 0.97, F1-score 0.89 → model sangat baik dalam mengidentifikasi penumpang yang tidak selamat.
- **Kelas 1 (Selamat):** Precision 0.73, Recall 0.28, F1-score 0.40 → model kesulitan mendeteksi penumpang yang selamat (banyak yang terlewat / False Negative tinggi).
- Perbedaan performa antar kelas ini karena **data tidak seimbang** (kelas 0 jauh lebih banyak dari kelas 1).

### 2. Makna TP, FP, TN, FN pada Kasus Titanic

| Istilah | Arti | Contoh |
|---|---|---|
| **True Positive (TP)** | Model memprediksi selamat, dan penumpang memang selamat | ✓ Tepat |
| **True Negative (TN)** | Model memprediksi tidak selamat, dan memang tidak selamat | ✓ Tepat |
| **False Positive (FP)** | Model memprediksi selamat, tapi penumpang tidak selamat | ✗ Alarm palsu |
| **False Negative (FN)** | Model memprediksi tidak selamat, tapi penumpang selamat | ✗ Terlewatkan |

### 3. Kesimpulan

Logistic Regression bekerja cukup baik untuk dataset Titanic dengan akurasi ~82%. Namun perlu diperhatikan bahwa model cenderung bias ke kelas mayoritas (kelas 0). Untuk meningkatkan performa pada kelas minoritas, dapat dicoba teknik seperti class weighting (`class_weight='balanced'`) atau oversampling (SMOTE).